## Стэнфордский курс CS 329a «Самосовершенствующиеся ИИ-агенты». Домашнее задание № 1

В этом домашнем задании мы рассмотрим основные методы масштабирования вычислений на этапе формирования ответа и самосовершенствования моделей.

Домашнее задание состоит из 5 частей:

1. Оценка без примеров (zero-shot) — 10 баллов
2. Голосование большинством — 30 баллов
3. Выбор лучшего из N ответов с помощью генеративной модели вознаграждения — 30 баллов
4. Самосовершенствование с использованием обратной связи — 45 баллов
5. Анализ результатов — 15 баллов





In [ ]:
%load_ext autoreload
%autoreload 2

## Импорт необходимых инструментов.
## Убедитесь, что пакет установлен согласно инструкции в README.md.
import os
import numpy as npcan
import matplotlib.pyplot as plt
from dotenv import load_dotenv

from cs329_hw1.tasks import AIME25
from cs329_hw1.methods import get_sampler, get_verifier

# Загрузка переменных окружения из файла .env
load_dotenv()

# Настройки
HF_TOKEN = os.getenv("HUGGINGFACE_HUB_TOKEN")
DEBUG_MODE = False  # True — во время разработки; False — для итоговых результатов

### Набор данных AIME25
Мы будем работать с набором данных AIME25 с платформы [Hugging Face](https://huggingface.co/datasets/math-ai/aime25).
Этот набор содержит 30 сложных математических задач Американского пригласительного математического экзамена (AIME).
Эти задачи значительно сложнее обычных школьных задач по математике и требуют продвинутых методов решения.
<br>
Сначала познакомимся с набором данных.


In [ ]:
aime25 = AIME25(hf_token=HF_TOKEN)
problems = aime25.get_problems(debug_mode=DEBUG_MODE)
system_prompt = aime25.get_system_prompt()

# Каждая задача представлена словарём с ключами "problem", "answer" и "id"
print(f"Loaded {len(problems)} AIME25 problems")
print("Problem: ", problems[0]["problem"])
print("answer: ", problems[0]["answer"])

Здесь каждая задача представлена словарём с ключами `"problem"` и `"answer"`.
<br>
Значение `"problem"` содержит условие математической задачи, а `"answer"` — правильный ответ. Важно, что это не тест с вариантами ответа, поэтому ответ часто может представлять собой математическое выражение.



### Полезные инструменты
Чтобы помочь вам реализовать методы, рассматриваемые в этом домашнем задании, мы подготовили несколько полезных функций в модуле `cs329_hw1.methods`. Рекомендуем ознакомиться с этой функцией, а также с другими функциями, которые будут представлены далее: это поможет понять их возможности и способы применения при выполнении задания.


In [ ]:
# В модуле `methods` реализовано несколько полезных методов.

# Сначала рассмотрим простой метод генерации нескольких ответов.
# Метод возвращает список списков: каждому входному запросу соответствует список сгенерированных ответов.
# Этот метод будет основой для приёмов, которые предстоит реализовать.
# Ниже показан пример генерации трёх ответов модели together_ai/Qwen/Qwen3-Next-80B-A3B-Instruct при температуре 0.7.

method = get_sampler("sample_multiple", "together_ai/Qwen/Qwen3-Next-80B-A3B-Instruct", temperature=0.7, system_prompt=system_prompt)
prompts = ["What language model are you based on?"]
responses = method(prompts)
print("\n".join(responses[0]))


### Проверка на одной задаче

In [ ]:
# Этот метод позволяет получать предсказания для заданной задачи.
# Получим с помощью метода первое предсказание.
# Получим три предсказания для первой задачи и выведем их.
problem_prompts = [problems[0]["problem"]]
predictions = method(problem_prompts)[0]
print("Sample response: ".join(predictions))


### Проверка правильности ответов

In [ ]:
# Для набора данных AIME25 предоставлен инструмент проверки ответов.
# Проверяющий модуль AIME25 определяет, верно ли предсказание.
# Он разбирает предсказание и эталонный ответ, затем сравнивает итоговые математические выражения.
# Обратите внимание: из-за сложностей разбора выражений проверяющий модуль может ошибаться.
# В основном используется проверяющий модуль Qwen на основе [репозитория Qwen-2.5 MATH](https://github.com/QwenLM/Qwen2.5-Math/tree/main/evaluation).
# Если захотите разработать более точный проверяющий модуль, мы будем рады увидеть результат!

# Проверим предсказания, полученные для первой задачи.
verifier = get_verifier("aime25")

print("Correct answer: ", problems[0]["answer"])
for prediction in predictions:
    print("Last line of the prediction: ", prediction.split("\n\n")[-1])
    print("Is correct: ", verifier.verify(prediction, problems[0]["answer"]))

## Теперь ваша очередь!

### 1. Оценка предсказаний без примеров (10 баллов)

Сначала мы оценим точность ответов модели при генерации одного варианта для каждой задачи, не используя специальные методы увеличения вычислений на этапе формирования ответа. <br>
Для вычисления точности достаточно использовать определённые выше объекты `method` и `verifier`.

Что нужно выполнить:
- Напишите код в области между метками `TODO: YOUR CODE STARTS HERE` и `TODO: YOUR CODE ENDS HERE`.
- Укажите ниже точность полученных ответов.
- Создайте переменную `zero_shot_correctness` типа `List[int]`, которая понадобится далее для сравнения результатов.


In [ ]:
method = get_sampler("sample_multiple", "together_ai/Qwen/Qwen3-Next-80B-A3B-Instruct", temperature=0.7, n_samples=1, system_prompt=system_prompt)

test_problems = problems
zero_shot_correctness = []

### TODO: YOUR CODE STARTS HERE — начало вашего кода


### TODO: YOUR CODE ENDS HERE — конец вашего кода
### Ниже выведите точность предсказаний.
print(f"Accuracy: {sum(zero_shot_correctness) / len(zero_shot_correctness)}")

### 2. Голосование большинством (30 баллов)

Здесь мы реализуем первый метод увеличения вычислений на этапе формирования ответа — голосование большинством, описанное в [этой статье](https://arxiv.org/abs/2408.03314) или в [более ранней работе](https://arxiv.org/abs/2203.11171). В частности:
- Для каждой задачи сгенерируйте несколько ответов (в данном случае — 16).
- Затем выберите ответ, набравший большинство голосов. Голосование проводится по нормализованным выражениям: из каждого полного решения выделяется итоговое числовое выражение, и голоса подсчитываются по этим выражениям. Для этого предоставлены вспомогательные функции.
- После этого сравните выбранный ответ с эталонным.

Что нужно выполнить:
- Напишите код в области между метками `TODO: YOUR CODE STARTS HERE` и `TODO: YOUR CODE ENDS HERE`.
- Укажите ниже точность полученных ответов.


In [ ]:
## 2.1 — Реализация голосования большинством.
from typing import List, Union
from cs329_hw1.methods.simple_samplers import SampleMultiple
from cs329_hw1.tasks.math_utils import (
    strip_string,
    extract_answer,
)

class MajorityVoting:
    """
    Класс реализует голосование большинством по нескольким вариантам ответа.
    Для каждого запроса он генерирует несколько ответов и выбирает наиболее частый.
    """

    def __init__(
        self,
        model: str,
        system_prompt: str = None,
        n_samples: int = 5,
        temperature: float = 0.7,
        max_workers: int = 16
    ):
        """
        Инициализирует метод голосования большинством.

        Аргументы:
            model (str): Название используемой модели.
            system_prompt (str, необязательно): Системный запрос для модели.
            n_samples (int, необязательно): Число ответов на один запрос; по умолчанию 5.
            temperature (float, необязательно): Температура генерации; по умолчанию 0.7.
        """
        self.sampler = SampleMultiple(
            model=model,
            system_prompt=system_prompt,
            n_samples=n_samples,
            temperature=temperature,
            max_workers=max_workers
        )

    def _parse_answer(self, response: str) -> str:
        return strip_string(extract_answer(response, "math"))

    def _get_majority_answer(self, responses: List[str]) -> str:
        """
        Определяет наиболее частый ответ с помощью простого подсчёта голосов.

        Аргументы:
            responses (List[str]): Список ответов модели.

        Возвращает:
            str: Наиболее частый ответ.
        """
        assert isinstance(responses, list), "Responses must be a list"
        assert all(
            isinstance(r, str) for r in responses
        ), "All responses must be strings"
        # Извлечение итоговых ответов из ответов модели

        ## TODO: YOUR CODE STARTS HERE — начало вашего кода
        ## Здесь реализуйте алгоритм голосования большинством.
        ## При необходимости используйте готовый метод `_parse_answer`.
        # Создайте счётчик для каждого уникального ответа
        
        ## TODO: YOUR CODE ENDS HERE — конец вашего кода
        return majority_answer

    def __call__(self, prompts: Union[str, List[str]], majority_voting_levels: List[int] = [1, 2, 4, 8, 16]) -> List[str]:
        """
        Выполняет голосование большинством для заданных запросов.

        Аргументы:
            prompts (str или List[str]): Один запрос или список запросов.

        Возвращает:
            List[List[str]]: Наиболее частый ответ для каждого уровня голосования.
        """
        
        majority_answers = []
        responses = self.sampler(prompts)
        for majority_voting_level in majority_voting_levels:
            majority_answers.append([self._get_majority_answer(responses[i][:majority_voting_level]) for i in range(len(responses))])
        return majority_answers

In [ ]:
method = MajorityVoting(
    model="together_ai/Qwen/Qwen3-Next-80B-A3B-Instruct",
    n_samples=16,
    temperature=0.7,
    system_prompt=system_prompt,
    max_workers=32
)

In [ ]:
## Не изменяйте код ниже: он нужен для оценки точности при разном числе ответов, участвующих в голосовании большинством.

test_problems = problems

total = len(test_problems)
prompts = [problem["problem"] for problem in test_problems]

majority_voting_levels = [1, 2, 4, 8, 16]
# Получаем все ответы сразу (внутри метода используется многопоточность)
all_responses_list = method(prompts)


majority_voting_accuracies = []
for responses_list, majority_voting_level in zip(all_responses_list, majority_voting_levels):
    correct = 0
    
    for prediction, problem in zip(responses_list, test_problems):
        # Здесь не нормализуем предсказание повторно: оно уже нормализовано на этапе голосования большинством.
        # Поэтому устанавливаем параметр normalize_prediction в False.
        if verifier.verify(prediction, problem["answer"], normalize_prediction=False):
            correct += 1
        
    accuracy = correct / len(test_problems)
    majority_voting_accuracies.append(accuracy)


In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter([1], [sum(zero_shot_correctness) / len(zero_shot_correctness)], s=100, color="darkorange", marker="o", label='No test-time compute')
plt.plot(majority_voting_levels, majority_voting_accuracies, marker='o', color="green", label='Majority Voting')
plt.xlabel('Test-time Budget (# of API calls)')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Number of Samples in Majority Voting')
plt.legend()
plt.grid(True)
plt.show()

### 3. Выбор лучшего из N ответов с помощью генеративной модели вознаграждения (30 баллов)

Здесь мы реализуем второй метод увеличения вычислений на этапе формирования ответа — выбор лучшего из N вариантов с использованием генеративной модели вознаграждения. В частности:
- Для каждой задачи сгенерируйте несколько ответов (в данном случае — 16).
- Затем используйте большую языковую модель (LLM), чтобы сопоставить эти ответы и выбрать лучший. Это похоже на выбор лучшего из N вариантов с помощью модели вознаграждения, как в [этой статье](https://arxiv.org/abs/2408.03314), но вместо дискриминативной модели вознаграждения мы используем LLM в роли оценщика. Такой подход иногда называют [генеративной проверкой](https://arxiv.org/abs/2408.15240) или [генеративными моделями вознаграждения](https://arxiv.org/abs/2410.12832).
- Затем проверьте выбранный ответ, сравнив его с эталонным.
- **Дополнительное задание (2 балла):** мы предоставляем исходный промпт для генеративной модели вознаграждения. Если вы улучшите этот промпт и повысите точность метода относительно результатов со стандартным промптом, получите 2 дополнительных балла. Обязательного порога улучшения нет, но желательно добиться прироста примерно на 5 процентных пунктов.

Что нужно выполнить:
- Напишите код в области между метками `TODO: YOUR CODE STARTS HERE` и `TODO: YOUR CODE ENDS HERE`.
- Представьте ниже график.


In [ ]:
class LLMVoting:
    """
    Класс использует LLM для объединения нескольких ответов и выбора лучшего.
    Для каждого запроса он генерирует несколько ответов и выбирает лучший отдельным вызовом LLM.
    """

    def __init__(
        self,
        model: str,
        system_prompt: str = None,
        n_samples: int = 5,
        temperature: float = 0.7,
        max_workers: int = 16
    ):
        """
        Инициализирует метод голосования с помощью LLM.

        Аргументы:
            model (str): Название используемой модели.
            system_prompt (str, необязательно): Системный запрос для модели.
            n_samples (int, необязательно): Число ответов на один запрос; по умолчанию 5.
            temperature (float, необязательно): Температура генерации; по умолчанию 0.7.
        """
        self.sampler = SampleMultiple(
            model=model,
            system_prompt=system_prompt,
            n_samples=n_samples,
            temperature=temperature,
            max_workers=max_workers
        )
        
        self.aggregator = SampleMultiple(
            model=model,
            system_prompt=system_prompt + """\n\nYou are a mathematical expert. You will be shown multiple solutions to a math problem.
Your task is to analyze these solutions and select the most likely correct answer.""",
            n_samples=1,
            temperature=0,  
            max_workers=max_workers
        )

    def _create_aggregation_prompts(self, problems: List[str], all_responses: List[List[str]]) -> List[str]:
        """
        Создаёт запросы для параллельного объединения ответов.
        
        Аргументы:
            problems (List[str]): Список исходных задач.
            all_responses (List[List[str]]): Списки ответов для каждой задачи.
            
        Возвращает:
            List[str]: Список запросов для модели-агрегатора.
        """
        return [
            f"""Here is a math problem:
{problem}

I have received multiple solutions. Here they are:

{chr(10).join(f'Solution {i+1}:{chr(10)}{r}' for i, r in enumerate(responses))}

Based on these solutions, restate the solution here that you think is most likely correct."""
            for problem, responses in zip(problems, all_responses)
        ]

    def __call__(self, prompts: Union[str, List[str]]) -> List[List[str]]:
        """
        Выполняет голосование с помощью LLM для заданных запросов.

        Аргументы:
            prompts (str или List[str]): Один запрос или список запросов.

        Возвращает:
            List[List[str]]: Выбранные ответы и все сгенерированные варианты.
        """
        # Получение нескольких вариантов ответа для каждого запроса
        ## TODO: YOUR CODE STARTS HERE — начало вашего кода
        
        ## TODO: YOUR CODE ENDS HERE — конец вашего кода

In [ ]:
# Реализация и тестирование голосования с помощью LLM
llm_method = LLMVoting(
    model="together_ai/Qwen/Qwen3-Next-80B-A3B-Instruct",
    n_samples=16,
    temperature=0.7,
    system_prompt=system_prompt,
    max_workers=32
)

# Проверяем голосование LLM на тех же задачах
test_problems = problems
prompts = [problem["problem"] for problem in test_problems]

# Получаем предсказания метода голосования LLM
llm_predictions = llm_method(prompts)

# Вычисляем точность голосования LLM
correct_llm_voting = 0
total_llm_voting = len(test_problems)

for prediction, problem in zip(llm_predictions, test_problems):
    if verifier.verify(prediction[0], problem["answer"]):
        correct_llm_voting += 1

llm_voting_accuracy = correct_llm_voting / total_llm_voting
print(f"LLM Voting Accuracy: {llm_voting_accuracy:.3f}")
print(f"Correct: {correct_llm_voting}/{total_llm_voting}")




### 4. Итеративное самосовершенствование с использованием обратной связи от выполнения

В этом разделе реализуется система самосовершенствования, вдохновлённая подходом RLEF (обучение с подкреплением на основе обратной связи от выполнения), описанным в [работе Gehring и соавторов (2024)](https://arxiv.org/pdf/2410.02089). Система генерирует первоначальное решение, анализирует его недостатки, а затем создаёт улучшенное решение с учётом полученной обратной связи.

**Ключевая идея:** вместо генерации нескольких независимых вариантов мы создаём одно решение, после чего получаем адресную обратную связь и генерируем новый вариант. Это напоминает то, как люди учатся на ошибках и постепенно совершенствуют свои решения.

**Этапы процесса:**
1. **Первоначальная генерация:** создайте одно решение задачи.
2. **Этап критического анализа:** найдите ошибки в решении и сформулируйте конструктивную обратную связь.
3. **Повторная генерация:** создайте новое решение с учётом замечаний.
4. **Оценка:** сравните точность исходного и улучшенного решений.

**Требования:**
- Реализуйте систему критического анализа, использующую одну модель для генерации обратной связи.
- Реализуйте повторную генерацию решения на основе полученных замечаний.
- Проверьте систему на всех 30 задачах набора данных AIME25.
- Сравните точность до и после самосовершенствования.
- Проанализируйте эффективность подхода, основанного на обратной связи.


In [ ]:
class SelfImprovementSystem:
    """
    Класс реализует классический подход RLEF (обучение с подкреплением на основе обратной связи от выполнения).
    Для генерации решений и критики используется только модель Qwen.
    """

    def __init__(self, model: str, system_prompt: str = None, temperature: float = 0.7, max_workers: int = 8):
        # Основной генератор первоначальных решений
        self.generator = SampleMultiple(
            model=model,
            system_prompt=system_prompt,
            n_samples=1,
            temperature=temperature,
            max_workers=max_workers
        )
        
        # Модель-критик для формирования обратной связи
        critic_system_prompt = """You are an expert mathematical critic. Your job is to analyze mathematical solutions and provide constructive feedback.

When given a problem and its solution, analyze it for:
1. Logical errors and reasoning gaps
2. Mathematical mistakes and calculation errors
3. Conceptual misunderstandings
4. Missing steps or incomplete reasoning

Provide specific, actionable feedback that can help improve the solution. If the solution is correct, acknowledge that but still suggest potential improvements or alternative approaches.

Format your feedback as:
- **Issues Found:** [List specific problems]
- **Suggestions:** [How to fix the issues]
- **Overall Assessment:** [Brief summary]"""

        self.critic = SampleMultiple(
            model=model,
            system_prompt=critic_system_prompt,
            n_samples=1,
            temperature=0.3,  # Более низкая температура для более последовательной и точной критики
            max_workers=max_workers
        )
        
        # Генератор улучшенных решений с учётом обратной связи
        regenerator_system_prompt = """You are an expert mathematician who learns from feedback. When given a problem, your previous solution, and feedback about that solution, you should:

1. Carefully analyze the feedback provided
2. Identify the key issues that need to be addressed
3. Generate a new, improved solution that addresses the feedback
4. Ensure your new solution is logically sound and mathematically correct

Always acknowledge the feedback and explain how you've addressed the main issues in your new solution."""

        self.regenerator = SampleMultiple(
            model=model,
            system_prompt=regenerator_system_prompt,
            n_samples=1,
            temperature=temperature,
            max_workers=max_workers
        )

    def _generate_initial_solution(self, problem: str) -> str:
        """Генерирует первоначальное решение задачи."""
        ## TODO: YOUR CODE STARTS HERE — начало вашего кода
        
        ## TODO: YOUR CODE ENDS HERE — конец вашего кода

    def _critique_solution(self, problem: str, solution: str) -> str:
        """
        Анализирует решение и формирует обратную связь.
        
        Аргументы:
            problem (str): Исходная задача.
            solution (str): Решение, которое нужно проверить.
            
        Возвращает:
            str: Замечания и рекомендации по решению.
        """
        ## TODO: YOUR CODE STARTS HERE — начало вашего кода
        
        ## TODO: YOUR CODE ENDS HERE — конец вашего кода

    def _regenerate_solution(self, problem: str, original_solution: str, feedback: str) -> str:
        """
        Генерирует улучшенное решение на основе обратной связи.
        
        Аргументы:
            problem (str): Исходная задача.
            original_solution (str): Первоначальное решение.
            feedback (str): Замечания к первоначальному решению.
            
        Возвращает:
            str: Улучшенное решение.
        """
        ## TODO: YOUR CODE STARTS HERE — начало вашего кода
        
        ## TODO: YOUR CODE ENDS HERE — конец вашего кода

    def improve_solution(self, problem: str) -> dict:
        """
        Выполняет полный цикл самоулучшения RLEF для одной задачи.
        
        Аргументы:
            problem (str): Задача, которую необходимо решить.
            
        Возвращает:
            dict: Первоначальное решение, обратную связь и улучшенное решение.
        """
        ## TODO: YOUR CODE STARTS HERE — начало вашего кода
        
        
        ## TODO: YOUR CODE ENDS HERE — конец вашего кода

        return {
            "problem": problem,
            "original_solution": original_solution,
            "feedback": feedback,
            "improved_solution": improved_solution
        }

    def evaluate_improvement(self, results: dict, correct_answer: str) -> dict:
        """
        Оценивает, стало ли улучшенное решение лучше первоначального.
        
        Аргументы:
            results (dict): Результаты метода improve_solution.
            correct_answer (str): Правильный ответ на задачу.
            
        Возвращает:
            dict: Результаты оценки, включая сравнение правильности ответов.
        """
        ## TODO: YOUR CODE STARTS HERE — начало вашего кода
        
        ## TODO: YOUR CODE ENDS HERE — конец вашего кода
        return {
            "original_correct": original_correct,
            "improved_correct": improved_correct,
            "improvement": improvement, # логическое значение (bool)
            "correct_answer": correct_answer
        } 

In [ ]:
# Инициализация системы самоулучшения (RLEF)
# Эта система использует только модель Qwen для генерации, критики и повторной генерации решений
self_improvement_system = SelfImprovementSystem(
    model="together_ai/Qwen/Qwen3-Next-80B-A3B-Instruct",
    temperature=0.7,
    system_prompt=system_prompt,
    max_workers=16
)

test_problems_size=len(problems)
test_problems = problems

print(f"Testing RLEF self-improvement system on {test_problems_size} problems...")
print("This will take several minutes as we generate initial solutions, critique them, and regenerate improved solutions.")
print("Using only Qwen for all steps (generation, critique, regeneration).")
print()

# Сохраняем результаты для последующего анализа
improvement_results = []
original_accuracies = []
improved_accuracies = []

for i, problem in enumerate(test_problems):
    print(f"Processing problem {i+1}/{test_problems_size}...")
    
    # Запускаем полный процесс самоулучшения RLEF
    result = self_improvement_system.improve_solution(problem["problem"])
    
    # Оцениваем, удалось ли улучшить решение
    evaluation = self_improvement_system.evaluate_improvement(result, problem["answer"])
    
    # Сохраняем результаты
    improvement_results.append({
        "problem_index": i,
        "problem": problem["problem"],
        "correct_answer": problem["answer"],
        "result": result,
        "evaluation": evaluation
    })
    
    original_accuracies.append(evaluation["original_correct"])
    improved_accuracies.append(evaluation["improved_correct"])
    
    # Выводим ход выполнения
    if evaluation["improvement"]:
        print(f"  ✓ Improved from incorrect to correct!")
    elif evaluation["original_correct"] and evaluation["improved_correct"]:
        print(f"  → Both solutions correct")
    elif not evaluation["original_correct"] and not evaluation["improved_correct"]:
        print(f"  → Both solutions incorrect")
    else:
        print(f"  → Regressed from correct to incorrect")

print(f"\nCompleted processing {test_problems_size} problems!")

In [ ]:
# Анализируем результаты
print("=== RLEF Self-Improvement Analysis ===")
print()

# Вычисляем общие показатели точности
original_accuracy = sum(original_accuracies) / len(original_accuracies)
improved_accuracy = sum(improved_accuracies) / len(improved_accuracies)
improvement_count = sum(1 for i in range(len(original_accuracies)) 
                       if improved_accuracies[i] and not original_accuracies[i])
regression_count = sum(1 for i in range(len(original_accuracies)) 
                     if original_accuracies[i] and not improved_accuracies[i])

print(f"Original Solution Accuracy: {original_accuracy:.3f} ({sum(original_accuracies)}/{len(original_accuracies)})")
print(f"Improved Solution Accuracy: {improved_accuracy:.3f} ({sum(improved_accuracies)}/{len(improved_accuracies)})")
print(f"Accuracy Improvement: {improved_accuracy - original_accuracy:.3f}")
print(f"Problems Improved: {improvement_count}")
print(f"Problems Regressed: {regression_count}")
print()

# Показываем подробные примеры успешных улучшений
print("=== Detailed Improvement Examples ===")
improvement_examples = [r for r in improvement_results if r["evaluation"]["improvement"]]

if improvement_examples:
    print(f"Found {len(improvement_examples)} successful improvements. Showing first example:")
    example = improvement_examples[0]
    
    print(f"\nProblem: {example['problem']}")
    print(f"Correct Answer: {example['correct_answer']}")
    print(f"\nOriginal Solution:")
    print(example['result']['original_solution'][:500] + "..." if len(example['result']['original_solution']) > 500 else example['result']['original_solution'])
    
    print(f"\nFeedback from Qwen Critic:")
    print(example['result']['feedback'][:500] + "..." if len(example['result']['feedback']) > 500 else example['result']['feedback'])
    
    print(f"\nImproved Solution:")
    print(example['result']['improved_solution'][:500] + "..." if len(example['result']['improved_solution']) > 500 else example['result']['improved_solution'])
else:
    print("No successful improvements found in this subset.")

## Анализ и обсуждение результатов (15 баллов)

### Что необходимо представить

**Обязательно:** выполните следующие аналитические задания, опираясь на результаты экспериментов из предыдущих разделов.

### Задание 1. График сравнения эффективности методов (5 баллов)

Постройте общий сравнительный график, включающий:
1. Базовый результат без примеров (одна точка).
2. Точность голосования большинством при числе ответов [1, 2, 4, 8, 16].
3. Точность голосования с использованием LLM (одна точка для 16 ответов).
4. Точность метода самосовершенствования (одна точка, отражающая улучшение).

Требования:
- Используйте значения точности, вычисленные в предыдущих разделах.
- Добавьте понятные подписи, легенду и заголовок.
- Используйте разные цвета и маркеры для каждого метода.
- Установите подходящие границы осей и добавьте сетку.
- Примечание: метод самосовершенствования использует 3 обращения к API для каждой задачи (первоначальная генерация + критический анализ + повторная генерация).


In [ ]:
# Задание 1: график сравнения качества методов
# TODO: YOUR CODE STARTS HERE — начало вашего кода

# TODO: YOUR CODE ENDS HERE — конец вашего кода

## **Задание 2. Количественный анализ (5 баллов)**


На основе результатов экспериментов ответьте на следующие вопросы, подкрепив ответы конкретными числовыми данными:

1. **Ранжирование по точности:** расположите методы в порядке убывания точности. Приведите фактические значения точности для каждого метода.

**TODO**


2. **Эффективность вычислительных затрат:** рассчитайте прирост точности на каждый дополнительный вызов API для голосования большинством (при увеличении числа ответов с n=1 до n=16) и сравните его с самосовершенствованием (всего 3 вызова API). Какой метод обеспечивает большую отдачу от дополнительных вычислений?

**TODO**


3. **Анализ самосовершенствования:**
   - Насколько повышается точность благодаря самосовершенствованию по сравнению с ответами без примеров?
   - Судя по анализу категорий ошибок, какие ошибки чаще всего встречаются в неправильных решениях: логические, математические, концептуальные или связанные с неполнотой ответа?
   - Для какого процента задач удалось улучшить решение?

**TODO**


## **Задание 3. Выводы по отдельным методам (5 баллов)**


Ответьте на следующие вопросы о реализованных методах:

1. **Голосование большинством:**
   - При каком количестве сгенерированных ответов дальнейшее увеличение их числа начинает давать всё меньший эффект? Укажите момент, когда прирост точности становится минимальным.
   - Какова максимальная точность, достигнутая методом голосования большинством?

**TODO**

2. **Голосование с использованием LLM:**
   - Как точность голосования с использованием LLM соотносится с голосованием большинством при n=16?
   - Чем отличаются вычислительные затраты этих методов при n=16?

**TODO**



3. **Самосовершенствование:**
   - Как метод самосовершенствования соотносится с традиционными методами увеличения вычислений на этапе формирования ответа?
   - Каковы преимущества и недостатки повторной генерации с учётом обратной связи по сравнению с генерацией нескольких независимых решений?
   - Какие категории ошибок, по результатам ваших экспериментов, наиболее важны для эффективного улучшения решений?

**TODO**



